Cell 1-2: Setup

In [1]:
import os
import json
import time

import numpy as np
import pandas as pd

from dotenv import load_dotenv
from google import genai
from google.genai import types

In [2]:
load_dotenv("../.env")

GEMINI_MODEL = "gemini-3.8-flash"
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

df = pd.read_csv("../data/raw/fctr.csv", sep="\t")
print("Dataset shape:", df.shape)

Dataset shape: (3238, 7)


Cell 3: Load the embedding model

In [3]:
# sentence-transformers kütüphanesi, modeli Hugging Face’ten indiriyor
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

embedder = SentenceTransformer(EMBEDDING_MODEL)

# multilingual-e5-base'den gelen varsayılan 768
print("Embedding dimension:", embedder.get_embedding_dimension())

# Device modelin nerede çalıştığını gösteriyor: cpu ya da Apple Silicon Mac’lerde grafik işlemcisini kullanan mps
print("Device:", embedder.device)

/Users/simgecevik/Desktop/WT/teyit-asistani/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1019 22:23:15.749000 57104 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7088.46it/s]


Embedding dimension: 768
Device: mps:0


Cell 4: Your first embeddings

In [4]:
sentences = [
    "query: Amazon teslim edilmeyen paketleri çok ucuza mı satıyor?",
    "passage: Amazon'un sagiplerine ulaşamayan kargo paketlerini 1,95 euroya sattığı iddiası",
]

# normalize_embeddings=True -> Normalleştirmek, her vektörü uzunluğu 1 olacak şekilde ölçeklemek demek; yönü değişmez.
embeddings = embedder.encode(sentences, normalize_embeddings=True)

# 2 cümle, her biri 768 sayılık bir vektör
print("Shape: ", embeddings.shape)
print("First 5 numbers of the first vector:", embeddings[0][:5])

Shape:  (2, 768)
First 5 numbers of the first vector: [ 0.00249925  0.04273459 -0.02578337  0.01376207  0.00087566]


Cell 5: Semantic similarity experiment

In [5]:
query = "query: Amazon teslim edilmeyen paketleri çok ucuza mı satıyor?"

passages = [
    # Same claim, completely different wording
    "passage: Amazon'un sahiplerine ulaşamayan kargo paketlerini 1,95 euroya sattığı iddiası",
    # Shares the word "Amazon" but is about something else
    "passage: Amazon ormanlarında çıkan yangınları gösterdiği iddia edilen görüntüler",
    # Different topic, but also about e-commerce and fraud
    "passage: Sosyal medyada paylaşılan bir linkle sahte indirim kampanyası düzenlendiği iddiası",
    # Completely unrelated
    "passage: İstanbul'da selden okulların tatil edildiği iddiası",
]

query_vec = embedder.encode(query, normalize_embeddings=True)
passage_vecs = embedder.encode(passages, normalize_embeddings=True)


# With normalized vectors, cosine similarity is just the dot product
scores = passage_vecs @ query_vec

# sorted(zip(scores, passages), reverse=True) → pairs the scores with the passages and sorts them in descending order.
for score, text in sorted(zip(scores, passages), reverse=True):
    print(f"{score:.3f}  {text}")

0.860  passage: Amazon'un sahiplerine ulaşamayan kargo paketlerini 1,95 euroya sattığı iddiası
0.802  passage: Sosyal medyada paylaşılan bir linkle sahte indirim kampanyası düzenlendiği iddiası
0.791  passage: Amazon ormanlarında çıkan yangınları gösterdiği iddia edilen görüntüler
0.755  passage: İstanbul'da selden okulların tatil edildiği iddiası


Cell 6: Build a mini knowledge base

In [6]:
kb = df.head(10).copy()

# Each document = the fact-checked claim + the editors' summary of the findings
kb["summary"] = kb["summary"].fillna("")
kb["document"] = kb["claim"] + ". " + kb["summary"]

doc_vecs = embedder.encode(
    ["passage: " + text for text in kb["document"]],
    normalize_embeddings=True,
)

print("Knowledge base:", doc_vecs.shape)
kb[["claim_id", "claim", "label"]]


Knowledge base: (10, 768)


,claim_id,claim,label
0,2,Amazon'un Türkiye'de teslim edilmemiş ürünleri...,yanlış
1,5,Mickey Mouse’un penisiyle peynir deldiği film ...,yanlış
2,6,Kurtlar Vadisi'nde Milli Muharip Uçak'ın öngör...,yanlış
3,7,Basketbol topu şeklindeki binanın gerçek olduğ...,doğru
4,13,Threads'in logosunun Homer Simpson'ın kulaklar...,yanlış
5,14,ABD’de uçan araba testinin onaylandığı iddiası,doğru
6,15,İnegöl'deki Göçmen Sağlık Merkezi’nin yer aldı...,yanlış
7,18,Threads hesabını silen kullanıcıların Instagra...,doğru
8,28,Slovenya'da bir başhemşirenin üç farklı Covid-...,yanlış
9,29,Hollanda Hükümetinin otizmli ve hafif engellil...,yanlış


Cell 7: Retrieval function

In [7]:
def retrieve(claim, top_k=3):
    """Return the top_k most similar fact-checks for a user claim."""
    query_vec = embedder.encode("query: " + claim, normalize_embeddings=True)
    scores = doc_vecs @ query_vec

    # indices of the highest scores
    # np.argsort(scores) → returns the positions (indices) of the scores when they are sorted from smallest to largest.
    # [::-1] → reverses the list: from largest to smallest.
    # [:top_k] → takes the first top_k items.
    top_idx = np.argsort(scores)[::-1][:top_k]

    results = kb.iloc[top_idx].copy()
    results["score"] = scores[top_idx]
    return results[["claim_id", "claim", "label", "summary", "url", "date", "score"]]

retrieve("Amazon teslim edilmeyen paketleri çok ucuza mı satıyor?")



,claim_id,claim,label,summary,url,date,score
0,2,Amazon'un Türkiye'de teslim edilmemiş ürünleri...,yanlış,Amazon Türkiye’de teslim edilemeyen paketlerin...,https://teyit.org/analiz/amazonun-turkiyede-te...,11/07/2023,0.897257
5,14,ABD’de uçan araba testinin onaylandığı iddiası,doğru,İddiaya konu olan uçan araba Alef Aeronautics ...,https://teyit.org/analiz/abdde-alef-aeronautic...,08/07/2023,0.764477
8,28,Slovenya'da bir başhemşirenin üç farklı Covid-...,yanlış,İddialara konu olan videodaki kişi Ljubljana Ü...,https://teyit.org/analiz/slovenyada-bir-bashem...,04/07/2023,0.760855


Cell 8: Harder retrieval tests

In [8]:
print("Test 1: Fine-grained topic")
display(retrieve("Threads hesabımı silersem Instagram hesabım da silinir mi?"))

print("\nTest 2: Not in the knowledge base")
display(retrieve("Ankara'da ekim ayında kar yağdı mı?"))

Test 1: Fine-grained topic


,claim_id,claim,label,summary,url,date,score
7,18,Threads hesabını silen kullanıcıların Instagra...,doğru,Threads hesabını tamamen silmenin tek yolu 6 T...,https://teyit.org/analiz/threads-hesabini-sile...,06/07/2023,0.878804
4,13,Threads'in logosunun Homer Simpson'ın kulaklar...,yanlış,Homer Simpson’ın orijinal çizimlerindeki kulak...,https://teyit.org/analiz/threadsin-logosunun-h...,10/07/2023,0.809159
1,5,Mickey Mouse’un penisiyle peynir deldiği film ...,yanlış,İddia görüntü dijital içerik üreten kullanıcı ...,https://teyit.org/analiz/mickey-mouseun-penisi...,11/07/2023,0.785240



Test 2: Not in the knowledge base


,claim_id,claim,label,summary,url,date,score
9,29,Hollanda Hükümetinin otizmli ve hafif engellil...,yanlış,Otizm spektrum bozukluğu ve zihinsel engellile...,https://teyit.org/analiz/hollanda-hukumetinin-...,04/07/2023,0.735661
8,28,Slovenya'da bir başhemşirenin üç farklı Covid-...,yanlış,İddialara konu olan videodaki kişi Ljubljana Ü...,https://teyit.org/analiz/slovenyada-bir-bashem...,04/07/2023,0.730258
6,15,İnegöl'deki Göçmen Sağlık Merkezi’nin yer aldı...,yanlış,İnegöl’deki 2 No’lu Göçmen Sağlığı Merkezi ayn...,https://teyit.org/analiz/inegoldeki-gocmen-sag...,07/07/2023,0.729696


Cell 9: Prompt builder

In [9]:
def build_prompt(claim, evidence):
    """Build a RAG prompt from the user's claim and the retrieved fact-checks."""
    evidence_blocks = []
    
    # evidence.itertuples() → iterates through each row of the table in sequence; we access the columns of that row using row.claim.
    # enumerate(..., start=1) → assigns a number starting from 1 to each piece of evidence. This allows the LLM to say things like “Based on Evidence 1,” and enables us to know which source is being referenced.
    for i, row in enumerate(evidence.itertuples(), start=1):
        evidence_blocks.append(
            f"[Kanıt {i}]\n"
            f"İncelenen iddia: {row.claim}\n"
            f"Doğrulama kuruluşunun kararı: {row.label}\n"
            f"Doğrulama tarihi: {row.date}\n"          # NEW: when the fact-check was published
            f"Bulgular: {row.summary}\n"
            f"Kaynak: {row.url}"
        )
    # "\n\n".join(...) → joins the evidence blocks, leaving an empty line between them.
    evidence_text = "\n\n".join(evidence_blocks)

    return f"""Sen Türkçe iddiaları değerlendiren bir doğrulama (fact-check) asistanısın.

KURALLAR:
1. Kararını YALNIZCA aşağıdaki kanıtlara dayandır. Kendi bilgini kullanma.
2. Önce kanıtların kullanıcının iddiasıyla aynı konuyu ele alıp almadığını kontrol et.
   Hiçbir kanıt doğrudan ilgili değilse kararın NOT_ENOUGH_EVIDENCE olmalı.
3. Kullanıcının iddiası, incelenen iddianın tersini söylüyor olabilir.
   Bu durumda kararı buna göre ters çevir.
4. Açıklamayı Türkçe, en fazla 3 cümle olarak yaz ve hangi kanıta dayandığını belirt.
5. Açıklamada kullandığın doğrulamanın tarihini belirt; durum o tarihten sonra
   değişmiş olabilir.

KULLANICININ İDDİASI:
{claim}

KANITLAR:
{evidence_text}

Cevabını yalnızca şu JSON formatında ver:
{{"verdict": "TRUE" | "FALSE" | "NOT_ENOUGH_EVIDENCE",
  "explanation": "Türkçe kısa gerekçe",
  "evidence_ids": [kullandığın kanıt numaraları]}}"""



Cell 10: Ask Gemini and run the full RAG pipeline

In [10]:
from google.genai import errors


def ask_gemini(prompt, max_retries=5):
    """Send a prompt to Gemini and parse the JSON answer, retrying on temporary errors."""
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0,
                    response_mime_type="application/json",
                ),
            )
            return json.loads(response.text)

        except errors.APIError as e:
            detail = str(e)
            # A daily quota will not reset by waiting a few seconds: stop immediately
            if e.code == 429 and "PerDay" in detail:
                raise RuntimeError("Daily Gemini quota exhausted, try again tomorrow") from e

            if e.code == 429 or e.code >= 500:
                wait = 5 * 2 ** attempt
                print(f"  Gemini returned {e.code}, retrying in {wait}s...")
                print(f"  Detail: {detail[:300]}")   # NEW: show what the API actually said
                time.sleep(wait)
            else:
                raise

    raise RuntimeError(f"Gemini still unavailable after {max_retries} attempts")

def fact_check(claim, top_k=3):
    """Full RAG pipeline: retrieve evidence, build the prompt, generate a verdict."""
    evidence = retrieve(claim, top_k=top_k)   # R: Retrieval
    prompt = build_prompt(claim, evidence)     # A: Augmentation
    result = ask_gemini(prompt)                # G: Generation
    return result, evidence

Cell 11: Run the tests

In [11]:
test_claims = [
    # 1. Paraphrased version of a claim rated "yanlış" -> expected FALSE
    "Amazon teslim edilemeyen paketleri 2 euro gibi çok ucuz fiyatlara satıyormuş.",
    # 2. Opposite of the same claim -> expected TRUE (the model must flip the verdict)
    "Amazon Türkiye'nin sahipsiz paketleri ucuza sattığı bir uygulaması yok.",
    # 3. Paraphrased version of a claim rated "doğru" -> expected TRUE
    "Threads hesabını tamamen silmek için Instagram hesabını da silmek gerekiyor.",
    # 4. Not in the knowledge base -> expected NOT_ENOUGH_EVIDENCE
    "Ankara'da ekim ayında kar yağdı.",
]

for claim in test_claims:
    try:
        result, evidence = fact_check(claim)
    except RuntimeError as e:
        print("CLAIM  :", claim)
        print("SKIPPED:", e)
        print("=" * 80)
        continue

    print("CLAIM      :", claim)
    print("VERDICT    :", result["verdict"])
    print("EXPLANATION:", result["explanation"])
    print("EVIDENCE   :", result["evidence_ids"])
    print("TOP SCORE  :", round(float(evidence["score"].iloc[0]), 3))
    print("=" * 80)
    time.sleep(2)

CLAIM  : Amazon teslim edilemeyen paketleri 2 euro gibi çok ucuz fiyatlara satıyormuş.
SKIPPED: Daily Gemini quota exhausted, try again tomorrow
CLAIM  : Amazon Türkiye'nin sahipsiz paketleri ucuza sattığı bir uygulaması yok.
SKIPPED: Daily Gemini quota exhausted, try again tomorrow
CLAIM  : Threads hesabını tamamen silmek için Instagram hesabını da silmek gerekiyor.
SKIPPED: Daily Gemini quota exhausted, try again tomorrow
CLAIM  : Ankara'da ekim ayında kar yağdı.
SKIPPED: Daily Gemini quota exhausted, try again tomorrow


In [13]:
"""result, evidence = fact_check("Threads hesabını tamamen silmek için Instagram hesabını da silmek gerekiyor.")
print(result)"""

'result, evidence = fact_check("Threads hesabını tamamen silmek için Instagram hesabını da silmek gerekiyor.")\nprint(result)'

Cell 12: Compare with no-RAG

In [14]:
def fact_check_without_rag(claim):
    """Ask Gemini directly, with no retrieved evidence."""
    prompt = f"""Sen Türkçe iddiaları değerlendire bir doğrulama asistanısın.
Aşağıdaki iddianın doğru olup olmadığına karar ver.

İDDİA: {claim}

Cevabını yalnızca şu JSON formatında ver:
{{"verdict": "TRUE" | "FALSE" | "NOT_ENOUGH_EVIDENCE",
  "explanation": "Türkçe kısa gerekçe"}}"""
    return ask_gemini(prompt)

for claim in test_claims:
    result = fact_check_without_rag(claim)
    print("CLAIM      :", claim)
    print("VERDICT    :", result["verdict"])
    print("EXPLANATION:", result["explanation"])
    print("=" * 80)
    time.sleep(2)

RuntimeError: Daily Gemini quota exhausted, try again tomorrow